In [8]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import joblib
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

In [2]:
# Suppress Optuna logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [16]:
# Cell 2: Load data
X_train = joblib.load('../data/processed/X_train_scaled.pkl')
y_train = joblib.load('../data/processed/y_train.pkl')
X_train_raw = joblib.load('../data/processed/X_train.pkl')
feature_names = joblib.load('../data/processed/feature_names.pkl')

print(f"Data loaded: {X_train.shape}")
print(f"Data loaded: {X_train_raw.shape}")

Data loaded: (70000, 7)
Data loaded: (70000, 7)


In [4]:
# Cell 3: Define objective for XGBoost
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = xgb.XGBClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    
    return scores.mean()

In [5]:
# Cell 4: Optimize XGBoost
print("Optimizing XGBoost...")
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nXGBoost Best F1: {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}")

Optimizing XGBoost...


Best trial: 7. Best value: 0.606298: 100%|██████████| 50/50 [09:53<00:00, 11.87s/it]


XGBoost Best F1: 0.6063
Best params: {'n_estimators': 408, 'learning_rate': 0.011279678584654455, 'max_depth': 5, 'subsample': 0.9711782718084285, 'colsample_bytree': 0.8101751034036374, 'gamma': 0.0002551004724639824, 'reg_alpha': 2.5245803249050743e-06, 'reg_lambda': 0.00012892448140379773}


In [6]:
# Cell 5: Define objective for LightGBM
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    
    model = lgb.LGBMClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    
    return scores.mean()

In [7]:
# Cell 6: Optimize LightGBM
print("\nOptimizing LightGBM...")
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nLightGBM Best F1: {lgb_study.best_value:.4f}")
print(f"Best params: {lgb_study.best_params}")


Optimizing LightGBM...


Best trial: 44. Best value: 0.60669: 100%|██████████| 50/50 [07:18<00:00,  8.76s/it] 


LightGBM Best F1: 0.6067
Best params: {'n_estimators': 488, 'learning_rate': 0.014774075219978757, 'max_depth': 10, 'num_leaves': 26, 'subsample': 0.9439667622160223, 'colsample_bytree': 0.8090673829038711, 'reg_alpha': 0.01813939007687263, 'reg_lambda': 0.018186792030602927}


In [17]:
# Cell 7: Define objective for CatBoost
def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_seed': 42,
        'verbose': False
    }
    
    model = CatBoostClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_raw, y_train, cv=cv, scoring='f1', n_jobs=-1)
    
    return scores.mean()

In [18]:
# Cell 8: Optimize CatBoost
print("\nOptimizing CatBoost...")
cat_study = optuna.create_study(direction='maximize')
cat_study.optimize(cat_objective, n_trials=50, show_progress_bar=True)

print(f"\nCatBoost Best F1: {cat_study.best_value:.4f}")
print(f"Best params: {cat_study.best_params}")


Optimizing CatBoost...


Best trial: 11. Best value: 0.606208: 100%|██████████| 50/50 [30:45<00:00, 36.91s/it]


CatBoost Best F1: 0.6062
Best params: {'iterations': 115, 'learning_rate': 0.041941839418325404, 'depth': 10, 'l2_leaf_reg': 7.675806911497271}


In [19]:
# Cell 9: Save optimization results
optuna_results = {
    'xgb_best_params': xgb_study.best_params,
    'xgb_best_f1': xgb_study.best_value,
    'lgb_best_params': lgb_study.best_params,
    'lgb_best_f1': lgb_study.best_value,
    'cat_best_params': cat_study.best_params,
    'cat_best_f1': cat_study.best_value
}

joblib.dump(optuna_results, '../models/optuna_best_params.pkl')

print("\nOptimization complete. Results saved.")


Optimization complete. Results saved.
